# VitroVision — Grad-CAM++ Visualization
**Method:** Grad-CAM++ (Chattopadhay et al., 2018)  
**Target layer:** `model.conv_head` (EfficientNet-B0 final conv, 1280-ch)  
**aug_smooth=True** — average over augmented copies for smoother heatmap

In [ ]:
import sys, importlib.util
from pathlib import Path
from importlib.metadata import version as _pkg_version

if importlib.util.find_spec('pytorch_grad_cam') is None:
    print('pytorch-grad-cam ยังไม่ได้ติดตั้ง')
    print('ติดตั้งด้วย: pip install grad-cam')
    raise SystemExit('หยุด: ต้องติดตั้ง grad-cam ก่อน')

print(f'grad-cam version: {_pkg_version("grad-cam")}')

## 1 — Imports + Load Model

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import transforms

ROOT       = Path('..').resolve()
MODEL_PATH = ROOT / 'models' / 'final' / 'classifier.pt'
LABELS     = ['healthy', 'contaminated', 'dead']
IMG_SIZE   = 224

# โหลด model
if not MODEL_PATH.exists():
    raise SystemExit(f'ไม่พบ model: {MODEL_PATH}')

model = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)
model.eval()
print(f'โหลด model จาก: {MODEL_PATH.name}')
print(f'conv_head: {model.conv_head}')

# Transform สำหรับ preprocessing (ไม่ใช้ augmentation)
_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 2 — Target Layer

In [ ]:
# conv_head คือ 1x1 Conv สุดท้ายก่อน classifier ใน EfficientNet-B0
# output: (B, 1280, H, W) — spatial resolution ดีที่สุดก่อน global pool
target_layers = [model.conv_head]

print(f'Target layer : model.conv_head')
print(f'Out channels : {model.conv_head.out_channels}')

## 3 — Run Grad-CAM++ บนภาพตัวอย่าง

In [ ]:
# ใช้ preview_orig.jpg ที่มีอยู่ใน shelf_manager/
SAMPLE_PATH = ROOT / 'shelf_manager' / 'preview_orig.jpg'

if not SAMPLE_PATH.exists():
    raise SystemExit(f'ไม่พบ: {SAMPLE_PATH}')

def run_gradcam(image_path, true_label=None):
    """
    คืน dict: rgb_orig, heatmap, overlay, pred_label, confidence
    image_path: str หรือ Path
    """
    bgr  = cv2.imread(str(image_path))
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Resize สำหรับ display (float32 ใน [0,1] สำหรับ show_cam_on_image)
    rgb_resized = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
    rgb_float   = rgb_resized.astype(np.float32) / 255.0

    # Tensor สำหรับ model
    tensor = _tf(rgb).unsqueeze(0)

    # Prediction
    with torch.no_grad():
        import torch.nn.functional as F
        probs = F.softmax(model(tensor), dim=1)[0]
    pred_idx    = probs.argmax().item()
    confidence  = probs[pred_idx].item()
    pred_label  = LABELS[pred_idx]

    # Grad-CAM++ — target = predicted class
    targets = [ClassifierOutputTarget(pred_idx)]
    with GradCAMPlusPlus(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=tensor, targets=targets,
                            aug_smooth=True, eigen_smooth=False)[0]

    overlay = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

    return {
        'rgb_orig':   rgb_resized,
        'heatmap':    grayscale_cam,
        'overlay':    overlay,
        'pred_label': pred_label,
        'confidence': confidence,
        'true_label': true_label,
        'path':       Path(image_path),
    }


result = run_gradcam(SAMPLE_PATH)

# 3-panel: Original / Heatmap / Overlay
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(result['rgb_orig'])
axes[0].set_title('Original', fontsize=12)

axes[1].imshow(result['heatmap'], cmap='jet')
axes[1].set_title('Grad-CAM++ Heatmap', fontsize=12)

axes[2].imshow(result['overlay'])
conf_pct = f"{result['confidence']:.1%}"
axes[2].set_title(f"Overlay — pred: {result['pred_label']} ({conf_pct})", fontsize=12)

for ax in axes: ax.axis('off')
plt.suptitle(result['path'].name, fontsize=10, color='gray')
plt.tight_layout()
plt.show()

print(f'Predicted : {result["pred_label"]} ({result["confidence"]:.3f})')

## 4 — Batch Visualization

In [ ]:
def visualize_sample(image_path, true_label=None, ax_row=None):
    """แสดง 3-panel สำหรับ 1 ภาพ บน ax_row=[ax0, ax1, ax2] หรือสร้าง figure ใหม่"""
    res = run_gradcam(image_path, true_label)

    if ax_row is None:
        fig, ax_row = plt.subplots(1, 3, figsize=(12, 4))
        standalone = True
    else:
        standalone = False

    ax_row[0].imshow(res['rgb_orig']);     ax_row[0].set_title('Original')
    ax_row[1].imshow(res['heatmap'], cmap='jet'); ax_row[1].set_title('Heatmap')
    ax_row[2].imshow(res['overlay'])

    title = f"{res['pred_label']} ({res['confidence']:.1%})"
    if res['true_label']:
        match = '✓' if res['pred_label'] == res['true_label'] else '✗'
        title = f"{match} true:{res['true_label']} | pred:{res['pred_label']} ({res['confidence']:.1%})"
    ax_row[2].set_title(title, fontsize=9)

    for ax in ax_row: ax.axis('off')
    if standalone:
        plt.suptitle(res['path'].name, fontsize=10, color='gray')
        plt.tight_layout(); plt.show()
    return res


# รัน × 2 preview ที่มีอยู่
preview_images = [
    (ROOT / 'shelf_manager' / 'preview_orig.jpg', None),
    (ROOT / 'shelf_manager' / 'preview_aug.jpg',  None),
]
preview_images = [(p, lbl) for p, lbl in preview_images if p.exists()]

n = len(preview_images)
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
if n == 1: axes = [axes]   # แก้ shape เมื่อมีแค่ 1 ภาพ

batch_results = []
for (img_path, true_lbl), ax_row in zip(preview_images, axes):
    res = visualize_sample(img_path, true_lbl, ax_row=ax_row)
    batch_results.append(res)
    print(f'{img_path.name}: pred={res["pred_label"]} ({res["confidence"]:.3f})')

plt.tight_layout()
plt.show()

## 5 — Paper-Ready Export (300 DPI)

In [ ]:
EXPORT_PATH = ROOT / 'results' / 'gradcam_preview.png'
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

n = len(batch_results)
fig, axes = plt.subplots(n, 3, figsize=(10, 3.5 * n), dpi=300)
if n == 1: axes = [axes]

col_titles = ['Original', 'Grad-CAM++ Heatmap', 'Overlay']

for row_idx, res in enumerate(batch_results):
    panels = [res['rgb_orig'], res['heatmap'], res['overlay']]
    cmaps  = [None, 'jet', None]
    for col_idx, (panel, cmap) in enumerate(zip(panels, cmaps)):
        ax = axes[row_idx][col_idx]
        ax.imshow(panel, cmap=cmap)
        ax.axis('off')
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=10, fontweight='bold')
    # row label
    conf_str = f"{res['pred_label']} ({res['confidence']:.1%})"
    axes[row_idx][0].set_ylabel(res['path'].name, fontsize=8, rotation=90,
                                 labelpad=4, va='center')

plt.suptitle('VitroVision — Grad-CAM++ (EfficientNet-B0, conv_head)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(str(EXPORT_PATH), dpi=300, bbox_inches='tight')
plt.show()

print(f'บันทึกแล้ว: {EXPORT_PATH}')
print(f'ขนาดไฟล์ : {EXPORT_PATH.stat().st_size / 1024:.1f} KB')